In [6]:
import os

from qdrant_client import QdrantClient


def create_qdrant_client() -> QdrantClient:
    qdrant_url = os.getenv("QDRANT_URL")

    if qdrant_url:
        return QdrantClient(
            url=qdrant_url,
            api_key=os.getenv("QDRANT_API_KEY") or None,
        )

    return QdrantClient(
        path=os.getenv(
            "QDRANT_PATH",
            "/home/ysh/workspace/coramail_agent/data/qdrant",
        )
    )

In [2]:
# QdrantClient 생성

client = create_qdrant_client()

print(client)

In [3]:
# 컬렉션 목록 확인

collections = client.get_collections()

print(collections)

collections=[]


In [4]:
# 사용 후 연결 종료

client.close()

### qdrant client에 collection 추가

In [7]:
from qdrant_client.models import Distance, VectorParams

EMAIL_COLLECTION = "coramail_emails"
ATTACHMENT_COLLECTION = "coramail_attachments"

VECTOR_SIZE = 1024


client = create_qdrant_client()


for collection_name in [
    EMAIL_COLLECTION,
    ATTACHMENT_COLLECTION,
]:
    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(
                size=VECTOR_SIZE,
                distance=Distance.COSINE,
            ),
        )


print(client.get_collections())

collections=[CollectionDescription(name='coramail_emails'), CollectionDescription(name='coramail_attachments')]


In [8]:
# 컬렉션 목록 확인

collections = client.get_collections()

print(collections)

collections=[CollectionDescription(name='coramail_emails'), CollectionDescription(name='coramail_attachments')]


Qdrant는 PostgreSQL처럼 컬렉션 생성 시 payload 필드를 선언하지 않습니다. Point를 넣을 때 payload 구조가 만들어집니다.

현재 CoRAMail의 메일·첨부파일 payload 구조를 축약 없이 반영한 예시입니다.

In [ ]:
from datetime import datetime, timezone
from qdrant_client.models import PointStruct

now = datetime.now(timezone.utc).isoformat()
dummy_vector = [0.01] * VECTOR_SIZE

email_point = PointStruct(
    id="11111111-1111-1111-1111-111111111111",
    vector=dummy_vector,
    payload={
        "point_type": "email",
        "email_id": "demo-email-001",
        "email_uid": "demo-email-001",
        "gmail_message_id": "",
        "gmail_thread_id": "",
        "rfc_message_id": "<demo-email-001@example.com>",
        "email_index": 0,
        "message_id": "<demo-email-001@example.com>",
        "thread_id": "납품 일정 확인",
        "in_reply_to": None,
        "references": [],
        "from": "customer@example.com",
        "to": "sales@coramail.example.com",
        "subject": "납품 일정 확인 요청",
        "subject_normalized": "납품 일정 확인 요청",
        "date_raw": "2026-07-23",
        "date_iso": "2026-07-23T09:00:00+09:00",
        "raw_text": "제품 입고 예정일을 확인 부탁드립니다.",
        "embedding_text": (
            "제목: 납품 일정 확인 요청\n"
            "보낸 사람: customer@example.com\n"
            "받는 사람: sales@coramail.example.com\n"
            "본문:\n제품 입고 예정일을 확인 부탁드립니다."
        ),
        "signature": "",
        "has_attachment": True,
        "attachment_count": 1,
        "attachment_ids": ["demo-attachment-001"],
        "mail_category": "기타",
        "business_refs": ["FM24291770"],
        "vessel_names": [],
        "content_hash": "demo-email-content-hash",
        "embedding_model": "nomic-embed-text",
        "created_at": now,
    },
)